In [8]:
# Install required packages (run once)
!pip install sentence-transformers xgboost scikit-learn pandas numpy joblib

Defaulting to user installation because normal site-packages is not writeable


In [9]:
import pandas as pd
import numpy as np
import re
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
from sentence_transformers import SentenceTransformer

print("✅ All imports successful")

✅ All imports successful


In [10]:
# =========================
# 1. Load Final Dataset
# =========================
# Update this path to match your system
DATA_PATH = r"D:/phishguard-extension/dataset/final_dataset.csv"

data = pd.read_csv(DATA_PATH)
data = data.dropna(subset=['text', 'label'])
data['text'] = data['text'].astype(str)
data['label'] = data['label'].astype(int)

print(f"Total samples : {len(data)}")
print(f"Label dist    :\n{data['label'].value_counts()}")
print(f"\nSample:\n{data.head(3)}")

Total samples : 45178
Label dist    :
label
1    22589
0    22589
Name: count, dtype: int64

Sample:
   label                                               text
0      1  Get the Finest Rolex Watch Replica ! Solid 14k...
1      0  Start Date: 5/1/01; HourAhead hour: 10; HourAh...
2      0  Purity of friendship between two is not about ...


In [11]:
# =========================
# 2. Engineered Features
#    (structural / syntactic signals)
# =========================

KEYWORDS = [
    'verify', 'urgent', 'login', 'password',
    'click', 'bank', 'otp', 'account', 'confirm',
    'suspended', 'security', 'alert', 'winner',
    'prize', 'free', 'limited', 'act now', 'expires',
    'credentials', 'validate', 'update your', 'kyc'
]

def extract_features(text: str) -> np.ndarray:
    text = str(text)
    tl   = text.lower()

    url_count          = len(re.findall(r'http[s]?://|www\.', text))
    special_char_ratio = len(re.findall(r'[^a-zA-Z0-9\s]', text)) / (len(text) + 1)
    digit_ratio        = len(re.findall(r'\d', text))           / (len(text) + 1)
    upper_ratio        = sum(1 for c in text if c.isupper())    / (len(text) + 1)
    exclaim_count      = text.count('!')
    word_count         = len(text.split())
    avg_word_len       = (sum(len(w) for w in text.split()) / (word_count + 1))

    keyword_flags = [1 if kw in tl else 0 for kw in KEYWORDS]

    return np.array(
        [url_count, special_char_ratio, digit_ratio, upper_ratio,
         exclaim_count, word_count, avg_word_len]
        + keyword_flags
    )

print("Building engineered features...")
X_engineered = np.array([extract_features(t) for t in data['text']])
print(f"Engineered feature matrix shape: {X_engineered.shape}")

Building engineered features...
Engineered feature matrix shape: (45178, 29)


In [12]:
# =========================
# 3. TF-IDF (lexical signal)
# =========================
print("Fitting TF-IDF...")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    stop_words='english',
    sublinear_tf=True          # dampens term-frequency to reduce outlier influence
)

X_tfidf = tfidf.fit_transform(data['text'])
print(f"TF-IDF matrix shape: {X_tfidf.shape}")

Fitting TF-IDF...
TF-IDF matrix shape: (45178, 10000)


In [ ]:
# =========================
# 4. Semantic Embeddings
#    Using all-MiniLM-L6-v2 (fast, 384-dim, multilingual-friendly)
#    This makes the model understand MEANING, not just word presence.
#
#    Why MiniLM?
#    - Designed for semantic similarity tasks
#    - 6x faster than BERT, fits in RAM easily
#    - Great at catching paraphrased phishing (e.g. "verify credentials" ≈ "confirm your details")
# =========================
print("Loading sentence-transformer (will download ~90 MB first time)...")

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

print("Encoding dataset — this may take a few minutes on CPU...")
texts = data['text'].tolist()

# batch_size=64 balances speed vs RAM; show_progress_bar gives ETA
X_semantic = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2-normalize so dot product = cosine sim
)

print(f"Semantic embedding matrix shape: {X_semantic.shape}")

# Save embedder for inference
embedder.save("phishguard_embedder")
print("✅ Embedder saved to 'phishguard_embedder/'")

Loading sentence-transformer (will download ~90 MB first time)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6017.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding dataset — this may take a few minutes on CPU...


Batches:   2%|▏         | 14/706 [10:29<43:51,  3.80s/it]    

In [ ]:
# =========================
# 5. Combine ALL Features
#    [TF-IDF (sparse)] + [Engineered (dense)] + [Semantic (dense)]
#    hstack requires all inputs to be sparse OR we convert dense to sparse
# =========================

X_sem_sparse = csr_matrix(X_semantic)
X_eng_sparse = csr_matrix(X_engineered)

X_final = hstack([X_tfidf, X_eng_sparse, X_sem_sparse])

print(f"Final combined feature matrix shape: {X_final.shape}")
# Expected: (45178, ~10384 + 384)  ≈ (45178, 10791)

Final combined feature matrix shape: (45178, 10413)


In [ ]:
# =========================
# 6. Train / Test Split
# =========================
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Train: 36142 | Test: 9036


In [ ]:
# =========================
# 7. Train XGBoost
#    tree_method='hist' is required for sparse + dense mixed input
# =========================
print("Training XGBoost...")

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.7,
    scale_pos_weight=1.0,
    eval_metric='logloss',
    tree_method='hist',       # fast histogram-based method — works with sparse input
    use_label_encoder=False,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print("\n✅ Training complete")

Training XGBoost...


C:\Users\hamza\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [14:28:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-logloss:0.63526
[50]	validation_0-logloss:0.12479
[100]	validation_0-logloss:0.06813
[150]	validation_0-logloss:0.05269
[199]	validation_0-logloss:0.04299

✅ Training complete


In [ ]:
# =========================
# 8. Threshold Tuning (F1-optimal)
# =========================
y_proba = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.2, 0.8, 0.05)
best_threshold = 0.5
best_f1 = 0

print(f"{'Threshold':>12} | {'F1':>8} | {'FN':>6} | {'FP':>6}")
print("-" * 40)

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    f1 = f1_score(y_test, y_pred)
    print(f"{t:12.2f} | {f1:8.4f} | {fn:6} | {fp:6}")

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"\n🏆 Best Threshold : {best_threshold:.4f}  |  F1 : {best_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, (y_proba >= best_threshold).astype(int)))

   Threshold |       F1 |     FN |     FP
----------------------------------------
        0.20 |   0.9860 |     32 |     95
        0.25 |   0.9873 |     35 |     80
        0.30 |   0.9879 |     40 |     70
        0.35 |   0.9884 |     42 |     63
        0.40 |   0.9898 |     43 |     49
        0.45 |   0.9901 |     45 |     44
        0.50 |   0.9905 |     48 |     38
        0.55 |   0.9907 |     51 |     33
        0.60 |   0.9908 |     54 |     29
        0.65 |   0.9906 |     62 |     23
        0.70 |   0.9899 |     70 |     21
        0.75 |   0.9891 |     82 |     16
        0.80 |   0.9882 |     95 |     11

🏆 Best Threshold : 0.6000  |  F1 : 0.9908

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4518
           1       0.99      0.99      0.99      4518

    accuracy                           0.99      9036
   macro avg       0.99      0.99      0.99      9036
weighted avg       0.99      0.9

In [ ]:
# =========================
# 9. Save All Artifacts
# =========================
joblib.dump(model,    "phishguard_model.pkl")
joblib.dump(tfidf,    "phishguard_tfidf.pkl")
joblib.dump(KEYWORDS, "phishguard_keywords.pkl")

with open("phishguard_threshold.txt", "w") as f:
    f.write(str(best_threshold))

# Note: embedder already saved in cell 4 via embedder.save()

print("\n✅ All artifacts saved:")
print("  phishguard_model.pkl")
print("  phishguard_tfidf.pkl")
print("  phishguard_keywords.pkl")
print("  phishguard_threshold.txt")
print("  phishguard_embedder/  (folder)")


✅ All artifacts saved:
  phishguard_model.pkl
  phishguard_tfidf.pkl
  phishguard_keywords.pkl
  phishguard_threshold.txt
  phishguard_embedder/  (folder)


In [ ]:
from sklearn.linear_model import SGDClassifier
import json

# =========================
# 10. Initialize Online Learner
#     Input to this model = [xgb_prob, fallback_score_placeholder]
#     It learns to CORRECT XGBoost's output from user feedback
# =========================

# Get XGBoost proba on test set as input to the online model
xgb_test_proba = model.predict_proba(X_test)[:, 1].reshape(-1, 1)

online_model = SGDClassifier(
    loss='log_loss',         # needed for predict_proba
    max_iter=1000,
    random_state=42
)

# Initial fit on XGBoost outputs (bootstrap)
online_model.partial_fit(xgb_test_proba, y_test, classes=[0, 1])

joblib.dump(online_model, "phishguard_online_model.pkl")
print("✅ Online model initialized and saved as 'phishguard_online_model.pkl'")

✅ Online model initialized and saved as 'phishguard_online_model.pkl'


In [ ]:
# =========================
# 11. Feedback Store (JSON)
#     Stores confirmed samples for batch retraining or audit
# =========================

FEEDBACK_FILE = "phishguard_feedback.json"

def save_feedback(text: str, label: int, prob: float):
    """Call this whenever a human confirms or corrects a prediction."""
    entry = {"text": text, "label": label, "prob": prob}

    if os.path.exists(FEEDBACK_FILE):
        with open(FEEDBACK_FILE, "r") as f:
            fb = json.load(f)
    else:
        fb = []

    fb.append(entry)

    with open(FEEDBACK_FILE, "w") as f:
        json.dump(fb, f, indent=2)

    print(f"Feedback saved (total: {len(fb)} entries)")

# Example usage:
# save_feedback("Click here to verify your bank account!", label=1, prob=0.91)

In [ ]:
# =========================
# 12. Incremental Retraining from Feedback
#     Run this periodically (e.g., daily cron, or triggered by API)
# =========================

def retrain_from_feedback():
    """Re-runs partial_fit on all feedback samples collected so far."""
    if not os.path.exists(FEEDBACK_FILE):
        print("No feedback data found.")
        return

    with open(FEEDBACK_FILE, "r") as f:
        fb = json.load(f)

    if not fb:
        print("Feedback store is empty.")
        return

    om = joblib.load("phishguard_online_model.pkl")
    embed = SentenceTransformer("phishguard_embedder")
    tf    = joblib.load("phishguard_tfidf.pkl")
    xgb   = joblib.load("phishguard_model.pkl")

    texts  = [e['text']  for e in fb]
    labels = [e['label'] for e in fb]

    # Rebuild full feature vector for each feedback sample
    X_sem = csr_matrix(embed.encode(texts, normalize_embeddings=True))
    X_tf  = tf.transform(texts)
    X_eng = csr_matrix(np.array([extract_features(t) for t in texts]))
    X_fb  = hstack([X_tf, X_eng, X_sem])

    # Get XGBoost probabilities
    fb_proba = xgb.predict_proba(X_fb)[:, 1].reshape(-1, 1)

    # Update online model
    om.partial_fit(fb_proba, labels, classes=[0, 1])
    joblib.dump(om, "phishguard_online_model.pkl")

    print(f"✅ Online model updated on {len(fb)} feedback samples.")

# Run it now on any existing feedback:
retrain_from_feedback()

NameError: name 'os' is not defined